In [1]:
"""
Importing all necessary packages for all tasks
"""
from tenpy.tools.process import mkl_set_nthreads, mkl_get_nthreads
import yfinance as yf
from datetime import datetime
from sklearn.preprocessing import MinMaxScaler
import scipy.linalg as la
from tenpy.models.model import CouplingModel, NearestNeighborModel, MPOModel, CouplingMPOModel
from tenpy.networks.site import SpinHalfSite,BosonSite
import numpy as np
from tenpy.tools.params import asConfig
from tenpy.models.lattice import Site, Chain, MultiSpeciesLattice
from tenpy.linalg import np_conserved as npc
import tenpy.models.spins
import tenpy.networks.mps as mps
import tenpy.networks.site as site
from tenpy.algorithms import tdvp
from tenpy.networks.mps import MPS
import scipy.special
import copy
import time
import random
import matplotlib.pyplot as plt
import pandas as pd
from joblib import Parallel, delayed, dump, load
import time
import cProfile
import pstats
import psutil
import csv, ast
from qutip import *
import seaborn as sns
from os import listdir
from os.path import join
from scipy.io import wavfile
import IPython.display as ipd
from librosa.feature import melspectrogram
from librosa import power_to_db
from librosa.effects import trim
from sklearn.model_selection import KFold
import warnings
from tenpy.tools.misc import TenpyInconsistencyWarning

tenpy.tools.optimization.set_level(3)
rng = np.random.default_rng(43)
np.random.seed(43)
random.seed(43)

In [2]:
"""
We define the new boson site with proper quadrature operators
"""
class NewBosonSite(Site):
  def __init__(self, Nmax=1, delta_t=1, conserve='N', filling=0.):
    # Defining the conserve values
    if not conserve:
        conserve = 'None'
    if conserve not in ['N', 'parity', 'None']:
        raise ValueError("invalid `conserve`: " + repr(conserve))
    # Local Dimension of each Boson site
    dim = Nmax + 1
    states = [str(n) for n in range(0, dim)]
    if dim < 2:
        raise ValueError("local dimension should be larger than 1....")
    b = np.zeros([dim, dim], dtype=np.float64)  # destruction/annihilation operator
    for n in range(1, dim):
        b[n - 1, n] = np.sqrt(n)
    bd = np.transpose(b)  # .conj() wouldn't do anything
    dA = np.sqrt(delta_t) * b
    dAd = np.sqrt(delta_t) * bd
    theta = np.pi/5 # quadrature phase
    Q = (np.exp(-1j*theta)*b + np.exp(1j*theta)*bd) # first quadrature
    P = (np.exp(1j*theta)*bd-np.exp(-1j*theta)*b) * (1j) # second quadrature
    QQ = np.dot(Q, Q)
    PP = np.dot(P, P)
    #---- Vacuum Projection
    P0 = np.zeros([dim, dim], dtype=np.float64)
    P0[0, 0] = 1.0
    # Number operator
    Ndiag = np.arange(dim, dtype=np.float64)
    N = np.diag(Ndiag)
    NN = np.diag(Ndiag**2)
    dN = np.diag(Ndiag - filling)
    dNdN = np.diag((Ndiag - filling)**2)
    # Operator sets
    ops = dict(B=b, Bd=bd,dA=dA,dAd=dAd, Q=Q, P=P,P0=P0, PP=PP, QQ=QQ, N=N, NN=NN, dN=dN, dNdN=dNdN)
    if conserve == 'N':
        chinfo = npc.ChargeInfo([1], ['N'])
        leg = npc.LegCharge.from_qflat(chinfo, range(dim))
    elif conserve == 'parity':
        chinfo = npc.ChargeInfo([2], ['parity_N'])
        leg = npc.LegCharge.from_qflat(chinfo, [i % 2 for i in range(dim)])
    else:
        leg = npc.LegCharge.from_trivial(dim)
    self.Nmax = Nmax
    self.conserve = conserve
    self.filling = filling
    Site.__init__(self, leg, states, sort_charge=True, **ops)
    self.state_labels['vac'] = self.state_labels['0']  # alias
    self.charge_to_JW_parity = np.array([0] * leg.chinfo.qnumber, int)  # trivial
  def __repr__(self):
    """Debug representation of self."""
    return "BosonSite({N:d}, {c!r}, {f:f})".format(N=self.Nmax,
                                                    c=self.conserve,
                                                    f=self.filling)


In [3]:
class AtomMirror(CouplingModel, MPOModel):
  def __init__(self, model_params, Driver):
    # 0) read out/set default parameters
    model_params = asConfig(model_params, "AtomMirror") # readout the set parameters of the system 
    delta_t = model_params.get('delta_t', 1) # for simplicity, we always take this as 1
    delay_steps = model_params.get('delay_steps', 1)
    Delta = model_params.get('Delta', 0) # Default: omega_L - omega = Delta, it is 0 for on resonant laser
    gamma = model_params.get('gamma', 1.) # Atom decaying rate
    omega = model_params.get('omega', 1.5) # Input strength
    phi = model_params.get('phi', 0.) # phase 
    max_photon = model_params.get('max_photon', 3) # max local photon site dimension
    hbar = 1 # convention 
    self.bc = 'finite' 
    # 1) charges of the physical leg. The only time that we actually define charges!
    leg = tenpy.linalg.np_conserved.LegCharge.from_trivial(2) # non defined charge. We are not interested in conservation
    # 2) onsite operators
    USE_PREDEFINED_SITE = False
    if not USE_PREDEFINED_SITE:
        Sp = [[0., 1.], [0., 0.]]
        Sm = [[0., 0.], [1., 0.]]
        Sx = np.array(Sp) + np.array(Sm)
        Sy = 1j*(np.array(Sp) - np.array(Sm))
        Sxx = np.dot(Sx,Sx)
        Syy = np.dot(Sy,Sy)
        Sz = [[1, 0.], [0., -1]]
        Id = [[1,0],[0,1]]
        # (Can't define Sx and Sy as onsite operators: they are incompatible with Sz charges.)
        # 3) local physical site
        system_site = Site(leg, ['up', 'down'], Sp=Sp, Sm=Sm, Sz=Sz, Sx=Sx, Sy=Sy, Sxx=Sxx,Syy=Syy)
    else:
        system_site = SpinHalfSite(conserve=None)
    # Define time-bin sites (each bin can hold N photon state)
    if not USE_PREDEFINED_SITE:
        time_bin_site = NewBosonSite(Nmax=max_photon,delta_t=delta_t,conserve=None)
    else:
        time_bin_site = BosonSite(Nmax=max_photon,conserve=None)  # Modify as needed for bosonic modes
    # We define an MPS order of sites 
    sites = [time_bin_site]*2 + [system_site] + [time_bin_site for _ in range(delay_steps)]
    tenpy.networks.site.set_common_charges([system_site, time_bin_site], new_charges='drop') # drop the charges
    # 4) lattice
    # Construct the initial MPS (vacuum state for time bins, initial state for the system)
    lattice = Chain(1, time_bin_site, bc="periodic", bc_MPS="finite") # Create the base lattice
    lat = MultiSpeciesLattice(lattice,sites)  # Create the actual lattice
    # 5) initialize CouplingModel
    CouplingModel.__init__(self, lat)
    # 6) add terms of the Hamiltonian
    # adding the atom hamiltonian
    self.add_onsite(-hbar*Delta/2*delta_t, 2, 'Sz')
    self.add_onsite(-hbar*Delta/2*delta_t, 2, 'Id')
    self.add_onsite(-hbar/2*omega*delta_t*Driver, 2, 'Sm') 
    self.add_onsite(-hbar/2*omega*delta_t*Driver, 2, 'Sp')
    # adding the coupling term
    self.add_coupling(-1j*hbar*np.sqrt(gamma/2)*np.exp(-1j*phi), 2-1, 'dA', 2, 'Sp',1,plus_hc=True) # add coupling between atom site and current k site
    self.add_coupling(-1j*hbar*np.sqrt(gamma/2), 2, 'Sp', 2+1, 'dA',1, plus_hc=True) # add coupling between atom site and past site k-delay
    # the `plus_hc=True` adds the h.c. term
    # 7) initialize H_MPO
    MPOModel.__init__(self, lat, self.calc_H_MPO()) # initialize MPO for this hamiltonian

In [4]:
class System:
    def __init__(self, model_params, engine_params):
        #---------------------------
        #  obtain model parameters
        #---------------------------
        self.model_params = asConfig(model_params, "AtomMirror")
        self.delta_t = model_params.get('delta_t') #numerical step
        self.tau = model_params.get('tau') # delay
        self.delay_steps = self.tau # since delta t = 1
        # real delay time to time bins
        self.phi = model_params.get('phi')
        # max photon
        self.photon_number = model_params.get('max_photon')
        #---------------------------
        #  obtain algoritm parameters
        #---------------------------
        # max bin and parameters for algorithm
        self.max_bin = model_params.get('max_bin')
        self.swap_steps = self.delay_steps-1 # Given the delay is l, we only need to swap l-1 times
        #---------------------------
        #  time dynamics
        #---------------------------
        self.time_counter = 0
        self.t_max = model_params.get('t_max')
        self.data_points = int(self.t_max/self.delta_t)
        # runtime option: fast or accurate
        self.engine_params = asConfig(engine_params, "Engine")
        self.option = engine_params.get('run_option')
        self.filename = engine_params.get('file_name')
        self.write_header = True
        #---------------------------
        #  Tasks to inject
        #---------------------------
        random.seed(1)
        self.task = [random.uniform(0, 2) for _ in range(int(self.data_points/5))]
        self.task = self.repeat_mask(self.task, 5) # But for each point, repeat t_renew times
        # initilize the state
        self.psi = self.initial_state()
        #---------------------------
        #  Data collection
        #---------------------------
        ### for atoms
        self.Es = []
        self.N_total = []
        self.Sx = []
        self.Sy = []
        self.Sxx = []
        self.Syy = []
        ### for time bins
        fixed_bin_index = [i for i in range(-(self.delay_steps),1)] # storing all values from current time down to coming back field
        PQ_keys = ["PQ" + str(i) for i in fixed_bin_index]
        self.operators = {key: [] for key in (PQ_keys)}

    def repeat_mask(self, values, N):
        return np.repeat(values, N)
        
    def initial_state(self):
        """We always initialize the states as vacuum"""
        # Define operators
        Sp = [[0., 1.], [0., 0.]]
        Sm = [[0., 0.], [1., 0.]]
        Sx = np.array(Sp) + np.array(Sm)
        Sy = 1j*(np.array(Sp)-np.array(Sm))
        Sxx = np.dot(Sx,Sx)
        Syy = np.dot(Sy,Sy)
        Sz = [[1, 0.], [0., -1]]
        Id = [[1, 0.], [0., 1]]
        # Initialize the lattice
        leg = tenpy.linalg.np_conserved.LegCharge.from_trivial(2)
        system_site = Site(leg, ['up', 'down'], Sp=Sp, Sm=Sm, Sz=Sz, Sx=Sx, Sy=Sy, Sxx=Sxx,Syy=Syy) # theoretically sx and sy are not compatible with sz
        time_bin_site = NewBosonSite(Nmax=self.photon_number,delta_t=self.delta_t,conserve=None)  # Modify as needed for bosonic modes
        sites = [time_bin_site]*2 + [system_site] + [time_bin_site for _ in range(self.delay_steps)]
        initial_state = ['vac']*2 + ['down'] + ['vac'] * (self.delay_steps)
        # Create the matrix product state
        psi = MPS.from_product_state(sites, initial_state, "finite")
        print("Initial state is initiated")
        return psi

    def execution(self, option):
        if option == 'speed' and self.delay_steps > self.max_bin:
            # 1. Swap to bring delayed bins near the system
            for i in reversed(range(self.swap_steps)): # going back from the delayed site
                self.psi.swap_sites(2+i+1) # the atom is at 2
            # 2. Apply the interaction unitary
            self.tdvp_engine.run() # MPO 
            # Generating the vacuum state on the leftmost
            Lshape = self.psi._B[-2].shape[0]
            Mshape = self.psi._B[-2].shape[1]
            Rshape = self.psi._B[-1].shape[2]
            data = np.random.rand(Lshape, Mshape, Rshape) * 0
            data[0,0,0] = 1 # ground state is assumed as 1.
            # Rebuild B matrices
            arr = npc.Array.from_ndarray_trivial(data, labels=['vL', 'p', 'vR'])
            # Add one vacuum site in the first and remove two last sites with one vacuum site
            Bs = self.psi._B[0:1] + self.psi._B[0:1] + self.psi._B[1:-2] + [arr]
            # Rebuild the sites
            sites = self.psi.sites[0:1] + self.psi.sites[0:1] + self.psi.sites[1:-1] 
            # Rebuild singular values
            Svs = self.psi._S[0:1] + self.psi._S[0:1]+ self.psi._S[1:-2] + self.psi._S[0:1]
            # New psi
            self.psi = MPS(sites, Bs, Svs, bc='finite', form='B', norm=1.0)
            # Swap back
            for i in range(self.swap_steps):
                self.psi.swap_sites(2+i+2)
            self.measure()
            self.psi.swap_sites(2) 
        else:
            # 1. Swap to bring delayed bins near the system
            for i in reversed(range(self.swap_steps)): 
                self.psi.swap_sites(2+i+1)
            # 2. Apply the interaction unitary
            self.tdvp_engine.run()
            self.psi = MPS(self.psi.sites[0:1] + self.psi.sites[0:1] + self.psi.sites[1:],
                    self.psi._B[0:1] + self.psi._B[0:1] + self.psi._B[1:],
                    self.psi._S[0:1] + self.psi._S[0:1]+ self.psi._S[1:], bc='finite', form='B', norm=1.0)
            for i in range(self.swap_steps):
                self.psi.swap_sites(2+i+2)
            self.measure()
            self.psi.swap_sites(2)
            self.delay_steps +=1
            self.model_params['delay_steps'] = self.delay_steps
    
    def measure(self):
        rho_i = self.psi.get_rho_segment([2+1]) # the system is located at 3    
        up_idx = self.psi.sites[2+1].state_labels['up'] # measure the excitation
        self.Es.append(rho_i[up_idx, up_idx]) # Population
        ### atom's quadratures
        self.Sx.append(self.psi.expectation_value('Sx',[2+1]))
        self.Sxx.append(self.psi.expectation_value('Sxx',[2+1]))
        self.Sy.append(self.psi.expectation_value('Sy',[2+1]))
        self.Syy.append(self.psi.expectation_value('Syy',[2+1]))
        ### Operator's quadratures
        for key, value_list in self.operators.items():
            if 0 == int(key[2:]): # For index 0, we measure the field on the left of atom which is at 2
                v = (self.psi.expectation_value('P',[2]), self.psi.expectation_value('Q',[2]))
                value_list.append(v)
            elif self.option == 'speed' and self.delay_steps >= self.max_bin:  # For other index, -1,-2,-3, we measure the sites for how many steps from the atom at index 3
                v = (self.psi.expectation_value('P',[3-int(key[2:])]), self.psi.expectation_value('Q',[3-int(key[2:])]))
                value_list.append(v)
            else: # Start applying special treatment for the speed
                v = (self.psi.expectation_value('P',[3-int(key[2:])]), self.psi.expectation_value('Q',[3-int(key[2:])]))
                value_list.append(v)
        self.save_large_csv(self.filename, self.Es, self.Sx, self.Sxx, self.Sy, self.Syy, self.operators)
        
    def run(self):
        Driver = self.task[self.time_counter]
        model = AtomMirror(self.model_params, Driver)
        self.tdvp_engine = tdvp.TwoSiteTDVPEngine(self.psi, model, self.engine_params)
        self.execution(self.option)
        for i in range(self.data_points-1):
            self.time_counter +=1
            Driver = self.task[self.time_counter]
            atom_mirror = AtomMirror(self.model_params, Driver)
            self.tdvp_engine = tdvp.TwoSiteTDVPEngine(self.psi, atom_mirror, self.engine_params)          
            self.execution(self.option)

    """
    Saving Files
    """
    def row_generator(self,Es, Sx, Sxx, Sy, Syy, operators):
        i = self.time_counter
        row = {
            'Es': Es[i],
            'Sx': Sx[i],
            'Sxx': Sxx[i],
            'Sy': Sy[i],
            'Syy': Syy[i],
        }
        for key, values in operators.items():
            row[key] = values[i]
        return row

    def save_large_csv(self, filename, Es, Sx, Sxx, Sy, Syy, operators):
        keys = ['Es', 'Sx', 'Sxx', 'Sy', 'Syy'] + list(operators.keys())
        with open(filename, 'a', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=keys) 
            if self.write_header:
                writer.writeheader()
                self.write_header = False
                pass
            row = self.row_generator(Es, Sx, Sxx, Sy, Syy, operators)
            writer.writerow(row)

In [5]:
# System Parameters
tau = 15 # Delay
delay_steps = tau
t_max = 200*5 # Total Time Evolution
Delta = 0 # Detuning
gamma = .1 # Decaying
omega = .15 # input strength
phi = np.pi/3 # phase
max_photon = 2 # amount of photons
max_bin = 100 # Maximal bins
delta_t = 1 # system time step
chi_max = 5

model_params = {
      'tau': tau,
      't_max': t_max,  
      'Delta': Delta,
      'gamma': gamma,
      'omega' : omega,
      'phi': phi,
      'max_photon': max_photon,
      'max_bin' : max_bin,
      'delta_t': delta_t, 
      'delay_steps': delay_steps, 
  }

tdvp_params = {
    'start_time': 0,
    'run_option': 'speed',
    'file_name' : 'test.csv',
    'dt': 1,
    'trunc_params': {
        'chi_max': chi_max,
        'svd_min': 1.e-10,
        'trunc_cut': None
    },
    'N_steps': 1,
    "max_N_sites_per_ring" : 100000
  }


obj = System(model_params, tdvp_params)
obj.run()


Initial state is initiated


KeyboardInterrupt: 